导入依赖

In [ ]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from matplotlib import pyplot as plt
from models.GhostNet import GhostNet
from models.GhostNet_exp_GhostModule import GhostNet as GhostNet_exp
from models.GhostNet_paper import GhostNet as GhostNet_paper
from models.GhostNet_paper_expSe import GhostNet as GhostNet_se
from models.cnn_model2 import PrecisionBalancedCNN

设置随机种子

In [ ]:
# 在训练开始前设置随机种子
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 512
# 在训练代码开头调用
set_seed(seed) # 42, 2025, 1024, 512, 100

导入数据集

In [ ]:
def preprocessing(tensor):
    if tensor.mean() > 0.5:
        tensor = 1 - tensor
    return tensor

In [ ]:
def tensor_transform(x):
    if x.shape[1] == 1:
        return x.repeat_interleave(3, dim= 1)
    elif x.shape[1] == 4:
        return x[:, :3, :, :]
    elif x.shape[1] == 3:
        return x
    else:
        raise ValueError('Invalid input shape')

tensor_rgb_transform = transforms.Lambda(tensor_transform)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(preprocessing),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [ ]:
mnist_db = datasets.MNIST(root= 'dataFolder/', train= True, transform= transform, download= True)
mnist_loader = DataLoader(mnist_db, batch_size= 128, shuffle= True)

In [ ]:
our_db = datasets.ImageFolder(root= 'dataFolder/num', transform= transform)
our_loader = DataLoader(our_db, batch_size= 128, shuffle= True)
len(our_loader)

In [ ]:
x, y = next(iter(our_loader))
plt.imshow(x[0].permute(1, 2, 0), cmap='gray')
plt.show()

模型训练

In [ ]:
def train_model(model, train_loader, in_ch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr= 0.001)
    model.train()

    losses = []
    for epoch in range(10):  # 训练10个周期
        correct = 0
        total = 0
        for idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            if in_ch == 3: x = tensor_rgb_transform(x)
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += x.shape[0]
            correct += (torch.argmax(out, dim= -1) == y).sum().item()
            losses.append(loss.item())
            if idx % 1000 == 0:
                print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))
                print('Accuracy: {:.2f}%'.format(correct * 100 / total))

    return model.state_dict(), losses

In [ ]:
# model_1C = GhostNet_exp(n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1C, mnist_loader, in_ch= 1)
# torch.save(state_dict, 'save_models/ghost/GhostNet_expGM_1C_MD_42.pth')
# print('GhostNet_exp_1C saved!')

# model_3C = GhostNet_exp(n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3C, mnist_loader, in_ch= 3)
# torch.save(state_dict, 'save_models/ghost/GhostNet_expGM_3C_MD_42.pth')
# print('GhostNet_exp_3C saved!')

# model_1C_B = GhostNet(n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1C_B, mnist_loader, in_ch= 1)
# torch.save(state_dict, 'save_models/ghost/GhostNet_1C_MD_100.pth')
# print('GhostNet_1C saved!')

# model_3C_B = GhostNet(n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3C_B, mnist_loader, in_ch= 3)
# torch.save(state_dict, 'save_models/ghost/GhostNet_3C_MD_100.pth')
# print('GhostNet_3C saved!')

# model_1C_C = GhostNet(n_classes= 10, in_ch= 1)
# state_dict, losses = train_model(model_1C_C, our_loader, in_ch= 1)
# torch.save(state_dict, 'save_models/OD/GhostNet_1C_OD_42.pth')
# print('GhostNet_1C saved!')

# model_3C_C = GhostNet(n_classes= 10, in_ch= 3)
# state_dict, losses = train_model(model_3C_C, our_loader, in_ch= 3)
# torch.save(state_dict, 'save_models/OD/GhostNet_3C_OD_42.pth')
# print('GhostNet_3C saved!')

# model_1C_D = GhostNet_paper(in_ch= 1)
# state_dict, losses = train_model(model_1C_D, mnist_loader, in_ch= 1)
# torch.save(state_dict, f'save_models/ghost/GhostNet_paper_1C_MD_{seed}.pth')
# print('GhostNet_1C saved!')

# model_3C_D = GhostNet_paper(in_ch= 3)
# state_dict, losses = train_model(model_3C_D, mnist_loader, in_ch= 3)
# torch.save(state_dict, f'save_models/ghost/GhostNet_paper_3C_MD_{seed}.pth')
# print('GhostNet_3C saved!')

model_1C_E = PrecisionBalancedCNN('ghost', 1)
state_dict, losses = train_model(model_1C_E, mnist_loader, in_ch= 1)
torch.save(state_dict, f'save_models/SimpleCNN/model_ghost_1C_MD_{seed}.pth')
print('GhostNet_1C saved!')

model_3C_E = PrecisionBalancedCNN('ghost', 3)
state_dict, losses = train_model(model_3C_E, mnist_loader, in_ch= 3)
torch.save(state_dict, f'save_models/SimpleCNN/model_ghost_3C_MD_{seed}.pth')
print('GhostNet_3C saved!')